In [ ]:
# Подключает Google Drive к виртуальной машине Colab.
from google.colab import drive
drive.mount('/content/drive')

# TODO: укажите имя папки на Google Drive, в которой сохранена распакованная
# папка задания, например 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# После подключения Google Drive этот код позволяет интерпретатору Python
# виртуальной машины Colab загружать из неё Python-файлы.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# Загружает набор данных CIFAR-10 на Google Drive,
# если он ещё не существует.
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Полносвязные нейронные сети

В этом упражнении реализуем полносвязные сети модульным способом. Для каждого слоя будут реализованы функции `forward` и `backward`. Функция `forward` получает входные данные, веса и другие параметры, а возвращает результат и объект `cache`, содержащий данные, необходимые для обратного прохода:

```python
def layer_forward(x, w):
  """Получает входные данные x и веса w."""
  # Выполняем вычисления...
  z = # ... некоторое промежуточное значение
  # Выполняем дополнительные вычисления...
  out = # результат

  cache = (x, w, z, out) # Значения, нужные для вычисления градиентов

  return out, cache
```

Функция обратного прохода получает производные сверху по графу и объект `cache`, а возвращает градиенты по входным данным и весам:

```python
def layer_backward(dout, cache):
  """
  Получает dout (производную функции потерь по выходам) и cache,
  затем вычисляет производную по входным данным.
  """
  # Распаковываем значения из cache
  x, w, z, out = cache

  # Используем значения из cache для вычисления производных
  dx = # Производная функции потерь по x
  dw = # Производная функции потерь по w

  return dx, dw
```

Реализовав таким образом набор слоёв, можно легко комбинировать их для построения классификаторов с различными архитектурами.

In [ ]:
# Выполняем начальную настройку.
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # задаём размер графиков по умолчанию
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Автоматически перезагружаем внешние модули.
# См. http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  """Возвращает относительную ошибку."""
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# Загружаем предобработанные данные CIFAR-10.

data = get_CIFAR10_data()
for k, v in list(data.items()):
  print(('%s: ' % k, v.shape))

# Аффинный слой: прямой проход

Откройте файл `cs231n/layers.py` и реализуйте функцию `affine_forward`.

После этого проверьте реализацию, запустив следующую ячейку:

In [ ]:
# Проверяем функцию affine_forward.

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * np.prod(input_shape)
weight_size = output_dim * np.prod(input_shape)

x = np.linspace(-0.1, 0.5, num=input_size).reshape(num_inputs, *input_shape)
w = np.linspace(-0.2, 0.3, num=weight_size).reshape(np.prod(input_shape), output_dim)
b = np.linspace(-0.3, 0.1, num=output_dim)

out, _ = affine_forward(x, w, b)
correct_out = np.array([[ 1.49834967,  1.70660132,  1.91485297],
                        [ 3.25553199,  3.5141327,   3.77273342]])

# Сравниваем полученный результат с эталонным. Ошибка должна быть порядка e-9 или меньше.
print('Testing affine_forward function:')
print('difference: ', rel_error(out, correct_out))

# Аффинный слой: обратный проход

Теперь реализуйте функцию `affine_backward` и проверьте её с помощью численной проверки градиента.

In [ ]:
# Проверяем функцию affine_backward.
np.random.seed(231)
x = np.random.randn(10, 2, 3)
w = np.random.randn(6, 5)
b = np.random.randn(5)
dout = np.random.randn(10, 5)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)

# Ошибка должна быть порядка e-10 или меньше.
print('Testing affine_backward function:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# Активация ReLU: прямой проход

Реализуйте прямой проход функции активации ReLU в функции `relu_forward` и проверьте реализацию, запустив следующий код:

In [ ]:
# Проверяем функцию relu_forward.

x = np.linspace(-0.5, 0.5, num=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = np.array([[ 0.,          0.,          0.,          0.,        ],
                        [ 0.,          0.,          0.04545455,  0.13636364,],
                        [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]])

# Сравниваем полученный результат с эталонным. Ошибка должна быть порядка e-8.
print('Testing relu_forward function:')
print('difference: ', rel_error(out, correct_out))

# Активация ReLU: обратный проход

Теперь реализуйте обратный проход функции активации ReLU в функции `relu_backward` и проверьте реализацию с помощью численной проверки градиента:

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10)
dout = np.random.randn(*x.shape)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

# Ошибка должна быть порядка e-12.
print('Testing relu_backward function:')
print('dx error: ', rel_error(dx_num, dx))

## Контрольный вопрос 1

В этом задании требуется реализовать только ReLU, однако в нейронных сетях можно использовать множество функций активации, каждая из которых имеет свои преимущества и недостатки. В частности, распространённая проблема функций активации --- нулевой или близкий к нулю поток градиента при обратном распространении ошибки. Какие из следующих функций активации имеют эту проблему? Если рассматривать эти функции в одномерном случае, какие типы входных данных приводят к такому поведению?

1. Sigmoid
2. ReLU
3. Leaky ReLU

$\color{blue}{\textit Ваш ответ:}$ *заполните здесь*

# Составные слои

В нейронных сетях часто встречаются типичные последовательности слоёв. Например, после аффинного слоя обычно следует нелинейность ReLU. Чтобы упростить работу с такими шаблонами, в файле `cs231n/layer_utils.py` определено несколько вспомогательных составных слоёв.

Сначала ознакомьтесь с функциями `affine_relu_forward` и `affine_relu_backward`, затем запустите следующую ячейку для численной проверки градиента обратного прохода:

In [ ]:
from cs231n.layer_utils import affine_relu_forward, affine_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 4)
w = np.random.randn(12, 10)
b = np.random.randn(10)
dout = np.random.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

# Относительная ошибка должна быть порядка e-10 или меньше.
print('Testing affine_relu_forward and affine_relu_backward:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# Слои потерь: Softmax

Теперь реализуйте функцию потерь и градиент Softmax в функции `softmax_loss` файла `cs231n/layers.py`. Они должны быть похожи на реализацию из `cs231n/classifiers/softmax.py`. Другие функции потерь, например `svm_loss`, также можно реализовать модульным способом, но в этом задании это не требуется.

Правильность реализации можно проверить, запустив следующую ячейку:

In [ ]:
np.random.seed(231)
num_classes, num_inputs = 10, 50
x = 0.001 * np.random.randn(num_inputs, num_classes)
y = np.random.randint(num_classes, size=num_inputs)


dx_num = eval_numerical_gradient(lambda x: softmax_loss(x, y)[0], x, verbose=False)
loss, dx = softmax_loss(x, y)

# Проверяем функцию softmax_loss. Значение функции потерь должно быть близко к 2.3,
# а ошибка dx --- порядка e-8.
print('\nTesting softmax_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))

# Двухслойная сеть

Откройте файл `cs231n/classifiers/fc_net.py` и завершите реализацию класса `TwoLayerNet`. Прочитайте код, чтобы понять API. Проверить реализацию можно, запустив ячейку ниже.

In [ ]:
np.random.seed(231)
N, D, H, C = 3, 5, 50, 7
X = np.random.randn(N, D)
y = np.random.randint(C, size=N)

std = 1e-3
model = TwoLayerNet(input_dim=D, hidden_dim=H, num_classes=C, weight_scale=std)

print('Testing initialization ... ')
W1_std = abs(model.params['W1'].std() - std)
b1 = model.params['b1']
W2_std = abs(model.params['W2'].std() - std)
b2 = model.params['b2']
assert W1_std < std / 10, 'First layer weights do not seem right'
assert np.all(b1 == 0), 'First layer biases do not seem right'
assert W2_std < std / 10, 'Second layer weights do not seem right'
assert np.all(b2 == 0), 'Second layer biases do not seem right'

print('Testing test-time forward pass ... ')
model.params['W1'] = np.linspace(-0.7, 0.3, num=D*H).reshape(D, H)
model.params['b1'] = np.linspace(-0.1, 0.9, num=H)
model.params['W2'] = np.linspace(-0.3, 0.4, num=H*C).reshape(H, C)
model.params['b2'] = np.linspace(-0.9, 0.1, num=C)
X = np.linspace(-5.5, 4.5, num=N*D).reshape(D, N).T
scores = model.loss(X)
correct_scores = np.asarray(
  [[11.53165108,  12.2917344,   13.05181771,  13.81190102,  14.57198434, 15.33206765,  16.09215096],
   [12.05769098,  12.74614105,  13.43459113,  14.1230412,   14.81149128, 15.49994135,  16.18839143],
   [12.58373087,  13.20054771,  13.81736455,  14.43418138,  15.05099822, 15.66781506,  16.2846319 ]])
scores_diff = np.abs(scores - correct_scores).sum()
assert scores_diff < 1e-6, 'Problem with test-time forward pass'

print('Testing training loss (no regularization)')
y = np.asarray([0, 5, 1])
loss, grads = model.loss(X, y)
correct_loss = 3.4702243556
assert abs(loss - correct_loss) < 1e-10, 'Problem with training-time loss'

model.reg = 1.0
loss, grads = model.loss(X, y)
correct_loss = 26.5948426952
assert abs(loss - correct_loss) < 1e-10, 'Problem with regularization loss'

# Ошибки должны быть порядка e-7 или меньше.
for reg in [0.0, 0.7]:
  print('Running numeric gradient check with reg = ', reg)
  model.reg = reg
  loss, grads = model.loss(X, y)

  for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    grad_num = eval_numerical_gradient(f, model.params[name], verbose=False)
    print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))

# Solver

Откройте файл `cs231n/solver.py` и ознакомьтесь с ним, чтобы понять API. Затем создайте экземпляр `Solver` для обучения `TwoLayerNet`, достигающей примерно `36%` точности на проверочной выборке.

In [ ]:
input_size = 32 * 32 * 3
hidden_size = 50
num_classes = 10
model = TwoLayerNet(input_size, hidden_size, num_classes)
solver = None

##############################################################################
# TODO: с помощью экземпляра Solver обучите TwoLayerNet, достигающую примерно #
# 36% точности на проверочной выборке.                                       #
##############################################################################

##############################################################################
#                           КОНЕЦ ВАШЕГО КОДА                                #
##############################################################################

# Отладка обучения

С заданными выше параметрами точность на проверочной выборке должна составить около 0.36. Это не очень хороший результат.

Один из способов понять причину --- построить во время оптимизации графики функции потерь и точности на обучающей и проверочной выборках.

Другой способ --- визуализировать веса, выученные первым слоем сети. В большинстве нейронных сетей, обученных на визуальных данных, веса первого слоя при визуализации обычно имеют заметную структуру.

In [ ]:
# Запустите эту ячейку, чтобы визуализировать функцию потерь и точность
# на обучающей и проверочной выборках.

plt.subplot(2, 1, 1)
plt.title('Функция потерь на обучении')
plt.plot(solver.loss_history, 'o')
plt.xlabel('Итерация')

plt.subplot(2, 1, 2)
plt.title('Точность')
plt.plot(solver.train_acc_history, '-o', label='обучение')
plt.plot(solver.val_acc_history, '-o', label='проверка')
plt.plot([0.5] * len(solver.val_acc_history), 'k--')
plt.xlabel('Эпоха')
plt.legend(loc='lower right')
plt.gcf().set_size_inches(15, 12)
plt.show()

In [ ]:
from cs231n.vis_utils import visualize_grid

# Визуализируем веса сети.

def show_net_weights(net):
    W1 = net.params['W1']
    W1 = W1.reshape(3, 32, 32, -1).transpose(3, 1, 2, 0)
    plt.imshow(visualize_grid(W1, padding=3).astype('uint8'))
    plt.gca().axis('off')
    plt.show()

show_net_weights(model)

# Подбор гиперпараметров

**В чём проблема?** По приведённым выше визуализациям видно, что функция потерь убывает более или менее линейно. Это может означать, что скорость обучения слишком мала. Кроме того, разрыва между точностью на обучающей и проверочной выборках нет, что указывает на недостаточную ёмкость модели, поэтому стоит увеличить её размер. С другой стороны, у очень большой модели ожидается более сильное переобучение, которое проявится в большом разрыве между точностью на обучающей и проверочной выборках.

**Настройка.** Подбор гиперпараметров и развитие интуиции о том, как они влияют на итоговое качество, занимает значительную часть работы с нейронными сетями, поэтому здесь важно получить практический опыт. Ниже поэкспериментируйте с различными значениями гиперпараметров, включая размер скрытого слоя, скорость обучения, число эпох и силу регуляризации. Можно также настроить затухание скорости обучения, но хорошее качество должно получиться и со значением по умолчанию.

**Ориентировочные результаты.** Следует достичь точности классификации выше 48% на проверочной выборке. Лучшая сеть авторов задания достигает более 52%.

**Эксперимент:** цель этого упражнения --- получить максимально хороший результат на CIFAR-10 с помощью полносвязной нейронной сети; 52% можно использовать как ориентир. Можно реализовать собственные подходы, например PCA для снижения размерности, dropout или дополнительные признаки для `Solver`.

In [ ]:
best_model = None



#################################################################################
# TODO: подберите гиперпараметры с помощью проверочной выборки. Сохраните       #
# лучшую обученную модель в best_model.                                         #
#                                                                               #
# Для отладки сети может быть полезна визуализация, подобная использованной     #
# выше: для плохо настроенной сети она будет качественно отличаться.            #
#                                                                               #
# Подбирать гиперпараметры вручную интересно, но можно написать код для         #
# автоматического перебора возможных комбинаций, как в предыдущих упражнениях. #
#################################################################################

################################################################################
#                           КОНЕЦ ВАШЕГО КОДА                                  #
################################################################################

# Проверьте модель

Запустите лучшую модель на проверочной и тестовой выборках. Точность на обеих выборках должна превышать 48%.

In [ ]:
y_val_pred = np.argmax(best_model.loss(data['X_val']), axis=1)
print('Validation set accuracy: ', (y_val_pred == data['y_val']).mean())

In [ ]:
y_test_pred = np.argmax(best_model.loss(data['X_test']), axis=1)
print('Test set accuracy: ', (y_test_pred == data['y_test']).mean())

In [ ]:
# Сохраняем лучшую модель.
best_model.save("best_two_layer_net.npy")

## Контрольный вопрос 2

После обучения классификатора на нейронной сети может оказаться, что точность на тестовой выборке значительно ниже точности на обучающей. Какими способами можно уменьшить этот разрыв? Выберите все подходящие варианты.

1. Обучать на большем наборе данных.
2. Добавить больше нейронов в скрытый слой.
3. Увеличить силу регуляризации.
4. Ничего из вышеперечисленного.

$\color{blue}{\textit Ваш ответ:}$

$\color{blue}{\textit Ваше объяснение:}$